# Scribble interpolation: where do a scribble's portraits fall along the target axis?

One notebook for the **gender** and the **age** interpolation scenarios. Set `SCENARIO` in the
next cell and run top to bottom. Nothing here is a metric: it generates portraits from one
scribble, places them on the target axis (gender: PCA of Woman vs Man; age: the 40 -> 79 axis),
shows them sorted along it, and draws the PCA scatter of the paper.

Needs a GPU, the SD packages and a finished run (`four_targets/run.sh full <scenario>`), or any
scribble file. Everything heavy is in `cdm.models.sd.interpolation`.

In [ ]:
SCENARIO   = "gender"          # "gender" | "age"
SCRIBBLE   = None              # path to a scribble PNG; None = the MLGD-F scribble of the run below
RUN_DIR    = None              # None = four_targets/outputs/full_<GenderInterpolation|AgeInterpolation>

N_EVAL     = 100               # portraits generated from the scribble
N_TARGET   = 300               # gender: target images in total (split evenly over the 4 groups)
N_PER_AGE  = 3                 # age: target images per age
AGE_MIN, AGE_MAX = 40, 79      # age: inclusive range; the axis runs from AGE_MIN to AGE_MAX
SEED       = 999
N_PICKS    = 16                # photos in the sorted grid
PICKS      = None              # None = evenly spaced; or your own ranks into the sorted order,
                               # e.g. [1, 4, 7, 17, 21, 36, 38, 45, 55, 60, 65, 69, 70, 71, 74, 75]
OUT_DIR    = None              # None = interpolation/outputs/<GenderInterpolation|AgeInterpolation>

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, torch
import matplotlib.pyplot as plt
from PIL import Image

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cdm").is_dir())
os.chdir(REPO); sys.path.insert(0, str(REPO))
from cdm.models.sd import interpolation as itp
from cdm.models.sd.models import load_models, make_deterministic
from cdm.models.sd.clip_utils import load_clip_model

NAME    = {"gender": "GenderInterpolation", "age": "AgeInterpolation"}[SCENARIO]
FT      = REPO / "experiments/stable_diffusion/four_targets"
RUN_DIR = Path(RUN_DIR) if RUN_DIR else FT / "outputs" / f"full_{NAME}"
OUT_DIR = Path(OUT_DIR) if OUT_DIR else REPO / "experiments/stable_diffusion/interpolation/outputs" / NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

def find(*names):   # first existing file among run_mlgd_f's and the paper's names
    for n in names:
        if (RUN_DIR / n).exists(): return RUN_DIR / n
    raise FileNotFoundError(f"none of {names} in {RUN_DIR}")

scribble_path = Path(SCRIBBLE) if SCRIBBLE else find("final_scribble_mlgd_f.png", "scribble_mlgdd.png")
scribble = Image.open(scribble_path).convert("RGB")
shown = [("scribble (MLGD-F)", scribble)]
for title, names in [("source portrait", ("source_portrait.png",)), ("source scribble", ("scribble.png", "scribble_source.png"))]:
    try: shown.insert(0, (title, Image.open(find(*names))))
    except FileNotFoundError: pass
fig, axes = plt.subplots(1, len(shown), figsize=(4 * len(shown), 4))
for ax, (t, im) in zip(np.atleast_1d(axes), shown): ax.imshow(im); ax.set_title(t); ax.axis("off")
plt.show()

make_deterministic(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
def pinned_revisions():   # four_targets/config/models.env -> {"architect_revision": sha, ...}
    out = {}
    for line in (FT / "config/models.env").read_text().splitlines():
        if "=" in line and not line.startswith("#"):
            k, v = line.split("=", 1)
            out[k.strip().lower()] = v.split("#")[0].strip()
    return out
rev = pinned_revisions()
architect, sprinter = load_models(device, architect_revision=rev.get("architect_revision"),
                                  sprinter_revision=rev.get("sprinter_revision"),
                                  controlnet_revision=rev.get("controlnet_revision"))
del architect; torch.cuda.empty_cache()
clip_model, clip_processor = load_clip_model(device, revision=rev.get("clip_revision"))
print(f"{NAME}: scribble {scribble_path}  ->  {OUT_DIR}")

## Target
Generated **on this scribble** (as the paper's interpolation notebooks did): the four gender
groups, or one group per age.

In [ ]:
if SCENARIO == "gender":
    groups = itp.GENDER_GROUPS
    target = itp.build_target(sprinter, clip_model, clip_processor, scribble, groups,
                              N_TARGET // len(groups), SEED, device)
else:
    ages   = list(range(AGE_MIN, AGE_MAX + 1))
    groups = [(str(a), itp.age_prompt(a)) for a in ages]
    target = itp.build_target(sprinter, clip_model, clip_processor, scribble, groups,
                              N_PER_AGE, SEED, device, seed_step=7)
print({g: e.shape[0] for g, e in target.items()})

## Portraits of the scribble

In [ ]:
photos, embs = itp.eval_photos(sprinter, clip_model, clip_processor, scribble, N_EVAL, SEED, device)
print(len(photos), "portraits,", embs.shape)

## The axis, and every portrait ranked along it
This sheet is for picking by eye: each tile shows its **rank** (0 = most feminine / youngest)
and its score. Copy ranks into `PICKS` in the first cell to fix the grid below.

In [ ]:
axis = itp.gender_axis(target) if SCENARIO == "gender" else itp.age_axis(target, AGE_MIN, AGE_MAX)
scores, coords = axis.project(embs)
fig = itp.ranked_strip(photos, scores, save_path=OUT_DIR / "ranked_all.png"); plt.show()

## Sorted grid (evenly spaced, or `PICKS`)

In [ ]:
fig, picks, idx = itp.sorted_grid(photos, scores, picks=PICKS, n_picks=N_PICKS,
                                  left=axis.left, right=axis.right,
                                  title=f"{NAME}: {N_PICKS} portraits along {axis.name}",
                                  save_path=OUT_DIR / "sorted_grid.png")
plt.show()
print("ranks shown:", picks)

## PCA scatter: target groups and the scribble's portraits

In [ ]:
if SCENARIO == "gender":
    fig = itp.scatter(axis, target, embs, save_path=OUT_DIR / "pca_scatter.png")
else:
    fig = itp.scatter(axis, target, embs, color_by={g: int(g) for g in target},
                      colorbar_label="Age", save_path=OUT_DIR / "pca_scatter.png")
plt.show()

## Save
`picks.json` (ranks, photo indices, scores, the full ranking) and `embeddings.npz` next to the
three PNGs, so the figure can be redrawn or the picks changed without regenerating.
`python experiments/stable_diffusion/export_results.py` copies the PNGs and `picks.json` into
`interpolation/results/<Scenario>/`.

In [ ]:
itp.save_run(OUT_DIR, NAME, str(scribble_path), picks, idx, scores, embs, target,
             extra={"seed": SEED, "n_eval": N_EVAL, "axis": axis.name})
print(sorted(os.listdir(OUT_DIR)))